# Report Selection: Random Date Generator

Generates random dates from 1 Jan 2015 onward to help pick the remaining reports for your 50-report sample. Reproducible via a fixed seed — document the seed value in your methodology so the exact same draw can be regenerated later if needed.

**How to use a date once generated:** the NSW Coroners Court findings pages are organised by year, not by exact date, so there's no direct "look up the report from this date" lookup. The intended workflow is: draw a date, browse that year's findings list, and take the report whose findings date is closest to (or on) the drawn date. This still needs a manual/search step per draw — the generator's job is just to make *which* date you start from a documented, reproducible random draw rather than an ad hoc choice.


## 1. The generator

In [1]:
import random
from datetime import date, timedelta


def random_date_generator(start: date = date(2015, 1, 1), end: date | None = None, seed: int | None = None):
    """Infinite generator of unique random dates in [start, end]. Reproducible: the same seed always produces the same sequence of dates, in the same order. End defaults to today if not given."""
    if end is None:
        end = date.today()
    if end <= start:
        raise ValueError("end must be after start")

    rng = random.Random(seed)
    span_days = (end - start).days
    seen = set()

    while True:
        offset = rng.randint(0, span_days)
        d = start + timedelta(days=offset)
        if d in seen:
            continue
        seen.add(d)
        yield d


## 2. Draw the dates you need, with a logged, reproducible seed

Pick a seed once and record it in your dissertation methods section — anyone re-running this cell with the same seed and the same `N_REPORTS_NEEDED` gets the identical sequence of dates back.


In [2]:
start_date = date(2015, 1, 1)
end_date = date(2026, 6, 30)

In [3]:
SEED = 2804  # fixed for reproducibility -- record this exact value in methodology
N_REPORTS_NEEDED = 22  # 3 for mhc/dd already annotated; remaining must be random

gen = random_date_generator(seed=SEED, start = start_date, end = end_date)
drawn_dates = [next(gen) for _ in range(N_REPORTS_NEEDED)]

for i, d in enumerate(drawn_dates, start=1):
    print(f"{i:>2}. {d.isoformat()}")

 1. 2021-08-25
 2. 2017-10-26
 3. 2015-04-20
 4. 2020-04-02
 5. 2015-03-14
 6. 2019-10-19
 7. 2024-11-16
 8. 2020-04-07
 9. 2015-05-16
10. 2021-05-09
11. 2016-08-31
12. 2023-09-18
13. 2015-08-11
14. 2026-03-29
15. 2021-09-07
16. 2018-02-14
17. 2023-04-10
18. 2015-04-04
19. 2018-11-06
20. 2017-12-06
21. 2017-07-13
22. 2016-04-10


## 3. Log the draw for your methodology appendix

Writes a small provenance file: the seed, the exact dates drawn, and when the draw was made. Append-friendly (a new run with the same seed produces the same dates and just re-confirms the log; a new seed adds a new record) so this file becomes your reproducibility record for report selection across the whole project.


In [4]:
import json
from datetime import datetime

log_record = {
    "drawn_at": datetime.now().isoformat(timespec="seconds"),
    "seed": SEED,
    "n_dates": len(drawn_dates),
    "dates": [d.isoformat() for d in drawn_dates],
}

with open("random_date_draws_20260927.jsonl", "a") as f:
    f.write(json.dumps(log_record) + "\n")

print(f"Logged {len(drawn_dates)} dates (seed={SEED}) to random_date_draws_20260927.jsonl")


Logged 22 dates (seed=2804) to random_date_draws_20260927.jsonl


# Inclusion and Exclusion Criteria



- Only reports from 01/01/2015 onwards are selected.
- Reports that include significant amounts of redaction are excluded from selection as crucial information may be missing and thus meaningful information cannot be extracted. If only names are redacted, these reports are considered useable.
- Reports which report on the death of multiple people, where the specific file number is not specified for each decedent, are excluded.
- Reports that are scans which require OCR are not selected for this task as the file URL requires a text layer in order for the LLM to work. 

Reports that were excluded:
- Report file number 2022/318454 for 29-04-2025 report is excluded as it is a scan and therefore doesn't have a text layer.
- For report date 23 February 2026, the first few reports are ones that were already annotated at the commencement of this project. The next suitable report is taken, which was file number 2023/171059.
- The first 3 reports for 2 April 2020 was a scanned document, which is excluded in this study. The next suitable report (file number 2014/240072) was used. 
- The first 2 reports of 14 March 2015 were not suitable (first one was scanned, second one was for multiple deaths), so the report with file number 2013/00055488 was used.
- The first report for 16 November 2024 is actually scanned and only the first page is text readable. The next report is used (file number 2019/0002334).
- 16 May 2015 first report is only 2 pages and includes a serious typo, which is excluded. The next report (file number: 2012/112395) is used. 
- 9 May 2021 first report is a scan so it's not used, the next report (file number 2017/168064)
- 31 August 2016 first few reports are scans so it's not used, the next useable report (file number 2014/00265393).
- 29 March 2026: numerous reports from this period were annotated when the project first began, so the next one available and appropriate for annotation was annotated here (file number: 2022/171534)

# Randomly assign reports to each dataset
There are 3 datasets the reports can belong to:
1. Prompt development (n = 5): these reports will be used to iteratively develop prompts to improve extraction performance of the model by human reviewing the outputs and fine-tuning the prompt. During prompt development, the model will extract the information 5 time (5 repeats) so that evaluation on the basis of accuracy and agreement can be conducted. 
2. Internal validation (n = 5): once the "best prompt" for zero-shot, few-shot, and chain-of-thought has been found, it will be tested on the internal validation dataset and evaluated. For internal validation, the model will extract the information 5 times (5 repeats) so that evaluation on the basis of accuracy and agreement can be conducted. The results of this will be used to decide what is the best prompt. 
3. External validation (n = 40): once the "best overall prompt" is found through evaluation, it will be used to extract information from the coronial reports and evaluated. 

To prevent bias and ensure fairness, the reports will be randomly assigned to either of these three datasets. 

In [ ]:
# code to randomly assign reports to each dataset